# Plant Disease

In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Dropout,Dense,Flatten
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
train_ds=tf.keras.utils.image_dataset_from_directory("/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/train",image_size=(224,224),batch_size=32,label_mode='categorical')
valid_ds=tf.keras.utils.image_dataset_from_directory("/kaggle/input/datasets/mohitsingh1804/plantvillage/PlantVillage/val",image_size=(224,224),batch_size=32,label_mode='categorical')

Found 43444 files belonging to 38 classes.


I0000 00:00:1783160049.061685      25 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783160049.064973      25 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 10861 files belonging to 38 classes.


In [3]:
class_names = train_ds.class_names

print(class_names)
print("Total Classes:", len(class_names))

['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Sp

In [4]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
])

In [5]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
valid_ds = valid_ds.map(lambda x, y: (normalization_layer(x), y))

In [6]:
model=Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)))
model.add(MaxPooling2D())

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(256,(3,3),activation='relu'))
model.add(MaxPooling2D())


model.add(Flatten())
model.add(Dense(512,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(38,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [8]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "PlantDisease.keras",
    monitor='val_accuracy',
    save_best_only=True
)

In [9]:
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/10


2026-07-04 10:14:23.938117: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-04 10:14:24.087033: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


   2/1358 ━━━━━━━━━━━━━━━━━━━━ 1:43 76ms/step - accuracy: 0.0547 - loss: 3.6563 

I0000 00:00:1783160067.688616      77 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1357/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.4826 - loss: 1.8993

2026-07-04 10:15:45.465511: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-04 10:15:45.611843: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1358/1358 ━━━━━━━━━━━━━━━━━━━━ 110s 75ms/step - accuracy: 0.6261 - loss: 1.3034 - val_accuracy: 0.8142 - val_loss: 0.6134
Epoch 2/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 86s 64ms/step - accuracy: 0.8233 - loss: 0.5611 - val_accuracy: 0.9038 - val_loss: 0.3063
Epoch 3/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 86s 63ms/step - accuracy: 0.8753 - loss: 0.3851 - val_accuracy: 0.9145 - val_loss: 0.2797
Epoch 4/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 86s 63ms/step - accuracy: 0.9085 - loss: 0.2823 - val_accuracy: 0.9207 - val_loss: 0.2474
Epoch 5/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 86s 63ms/step - accuracy: 0.9280 - loss: 0.2187 - val_accuracy: 0.9240 - val_loss: 0.2486
Epoch 6/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 87s 64ms/step - accuracy: 0.9426 - loss: 0.1772 - val_accuracy: 0.9409 - val_loss: 0.1960
Epoch 7/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 84s 62ms/step - accuracy: 0.9481 - loss: 0.1597 - val_accuracy: 0.9400 - val_loss: 0.2152
Epoch 8/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 85s 63ms/step - accuracy: 0.9547 - loss: 0.1

In [10]:
loss, accuracy = model.evaluate(valid_ds)

print("Validation Accuracy:", accuracy)

340/340 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.9551 - loss: 0.1542
Validation Accuracy: 0.9550685882568359
